In [ ]:
import os
try:
    path_initialized
except NameError:
    path_initialized = True
    os.chdir('..')

import numpy as np
from sympy.abc import x, y

# from qldpc import codes
import networkx as nx
import matplotlib.pyplot as plt
import stim
import sinter

import src.device as device
import src.plotting as plotter
from src.RotatedSurfaceCode import RotatedSurfaceCode
from src.HGPCode import HGPCode
from src.QECCode import TestCode
from src.decoders import BPOSD

In [ ]:
code_hgp = HGPCode.simplex_code(r=3)
hwp = device.default_hwp

num_qubits = len(code_hgp.data_indices + code_hgp.X_ancilla_indices + code_hgp.Z_ancilla_indices)
s = int(np.sqrt(len(code_hgp.data_indices)//2))
available_coords = set((x,y) for x in range(1, 2*s+1) for y in range(1, s+1))
data_coords_hgp = {}
for i,d in enumerate(code_hgp.data_indices):
    # coords = min(available_coords, key=lambda c: (c[0] - s/2)**2 + (c[1] - s/2)**2)
    # data_coords_hgp[d] = coords
    # available_coords.remove(coords)
    if d < s**2:
        data_coords_hgp[d] = (d//s+1, d%s+1)
    else:
        dd = d-s**2
        data_coords_hgp[d] = (s + dd%s+1, dd//s+1)

dev = device.UnitCellDevice(2*s+2, s+2, hwp)

In [ ]:
sched = dev.compile_QEC_schedule(code_hgp, data_coords_hgp, [], rounds=1, use_highways=True, refocus_shuttle_noise=False, optimize_ancilla_start=True, separate_X_Z=True)

In [ ]:
for i,c in data_coords_hgp.items():
    plt.text(c[0], c[1], str(i))
plt.xlim(0, 2*s+2)
plt.ylim(0, s+2)

Need ancilla qubits to be able to decide when and where they start the schedule. Maybe allow free move-anywhere-and-wait action that would be used to set the initialization spot? The wait would be allowed even in a blocked interval, but the end of the wait is an initialization that must be in an available interval.

Also need to let CX order be free



In [ ]:
sched.total_duration()

In [ ]:
frames = sched.get_frames(50)
plotter.plot_transition(dev, code_hgp, frames[50:70])

In [ ]:
frames = sched.get_frames(50)
plotter.plot_device_snapshot(dev, code_hgp, frames[10])
anim = plotter.animate_device(dev, code_hgp, frames, 'anim_simplex.gif', fps=10)